# SC15 - Multi-View Aggregation Ablation

An item has several photographed views. A shared ConvNeXt-S classifier predicts four tasks for each view, and pooling combines those predictions into one item decision. The current reference uses the **mean of per-view logits**. This notebook compares pooling rules using cached logits, without retraining the image backbone. The optional learned aggregators fit their own small models.

**Sources:** attach the original cached producer outputs or run where their `Results/` folders are visible. Cached logits are not bundled in the GitHub package.

- `kfold_convnext_small_logits.npz` from SC9: supplied results cover the historical 457-item pooled cohort; executed fold count is not verified by that archive.
- `logits_convnext_small.fb_in22k_ft_in1k.npz` from SC12: validation/test logits for learned aggregators. The validation set was already used in backbone development, so this is an exploratory comparison rather than an independent tuning evaluation.

**Methods:** `mean_logit`, `mean_prob`, `conf_weighted`, `best_view`, `max/noisyOR`, `median_logit`, `lse`, and optional learned attention/DeepSets.

**Metrics:** material accuracy/F1, route F1, threat QWK/MAE, condition AUROC, and threat-threshold recall/FPR. Threat decoding here uses CORN class-probability argmax, while shared headline metrics use cumulative-probability thresholds. Choosing a pooling rule using held-out scores is post-test selection. No completed SC15 export is supplied in this repository.


In [ ]:
import numpy as np, json, glob, os
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score
NUM_THREAT=4; HEADS=["material","condition","route","threat"]
OUT="/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def find(name):
    for root in ["/kaggle/input","Results","./Results","../Results","../input","."]:
        h=glob.glob(os.path.join(root,"**",name),recursive=True)
        if h: return h[0]
    raise FileNotFoundError(name+" -- attach the SC9/SC12 outputs as inputs")

KF  =np.load(find("kfold_convnext_small_logits.npz"),allow_pickle=True)
SC12=np.load(find("logits_convnext_small.fb_in22k_ft_in1k.npz"),allow_pickle=True)
print("loaded k-fold pool and SC12 val/test logits")


In [ ]:
def softmax(z): z=z-z.max(-1,keepdims=True); e=np.exp(z); return e/e.sum(-1,keepdims=True)
def sigmoid(z): return 1/(1+np.exp(-z))
def corn_cum(l): return np.cumprod(sigmoid(l),axis=-1)                 # P(y>k)
def cum_to_probs(cum):
    p=np.zeros(cum.shape[:-1]+(NUM_THREAT,)); p[...,0]=1-cum[...,0]
    for k in range(1,NUM_THREAT-1): p[...,k]=cum[...,k-1]-cum[...,k]
    p[...,-1]=cum[...,-1]; return np.clip(p,1e-6,1)
def corn_probs(l): return cum_to_probs(corn_cum(l))
def norm_entropy(p): return -(p*np.log(p+1e-9)).sum(-1)/np.log(p.shape[-1])

def group(D,split):
    items=D[f"{split}_items"]; log={h:D[f"{split}_log_{h}"] for h in HEADS}; ys={h:D[f"{split}_y_{h}"] for h in HEADS}
    uniq=list(dict.fromkeys(items.tolist())); g={it:[] for it in uniq}
    for j,it in enumerate(items): g[it].append(j)
    lab={h:np.array([ys[h][g[it][0]] for it in uniq]) for h in HEADS}
    return uniq,g,log,lab


In [ ]:
# ---- per-item aggregation: per-view logits -> item-level probability for one head ----
def agg_head(method, L, head):
    if head=="condition":
        pv=sigmoid(L)
        if   method=="mean_logit":   return sigmoid(L.mean())
        elif method=="mean_prob":    return pv.mean()
        elif method=="median_logit": return sigmoid(np.median(L))
        elif method=="best_view":    return pv[np.argmax(np.abs(pv-0.5))]
        elif method=="max_noisyOR":  return 1-np.prod(1-pv)
        elif method=="conf_weighted":
            w=np.abs(pv-0.5)+1e-6; return float((w*pv).sum()/w.sum())
        elif method=="lse":          return sigmoid((1/2.0)*np.log(np.exp(2.0*L).mean()))
    probs = corn_probs(L) if head=="threat" else softmax(L)
    if   method=="mean_logit":   return corn_probs(L.mean(0)) if head=="threat" else softmax(L.mean(0))
    elif method=="mean_prob":    return probs.mean(0)
    elif method=="median_logit": return corn_probs(np.median(L,0)) if head=="threat" else softmax(np.median(L,0))
    elif method=="best_view":    return probs[np.argmax(1-norm_entropy(probs))]
    elif method=="conf_weighted":
        w=(1-norm_entropy(probs))+1e-6; return (w[:,None]*probs).sum(0)/w.sum()
    elif method=="lse":
        tau=2.0; z=(1/tau)*np.log(np.exp(tau*L).mean(0)); return corn_probs(z) if head=="threat" else softmax(z)
    elif method=="max_noisyOR":
        if head=="threat":
            item_cum=1-np.prod(1-corn_cum(L),0); return cum_to_probs(item_cum)   # noisy-OR on P(y>k)
        m=probs.max(0); return m/m.sum()
    raise ValueError(method)

def aggregate(method, uniq, g, log, head):
    return np.array([agg_head(method, log[head][g[it]], head) for it in uniq])


In [ ]:
def m_material(p,y): pr=p.argmax(1); return dict(acc=accuracy_score(y,pr), f1=f1_score(y,pr,average="macro",labels=range(10),zero_division=0))
def m_route(p,y):    return dict(f1=f1_score(y,p.argmax(1),average="macro",labels=range(4),zero_division=0), acc=accuracy_score(y,p.argmax(1)))
def m_threat(p,y):
    pr=p.argmax(1); out=dict(qwk=cohen_kappa_score(y,pr,weights="quadratic",labels=list(range(4))), mae=float(np.abs(y-pr).mean()), acc=accuracy_score(y,pr))
    for T in (1,2,3):
        haz=(y>=T); nz=~haz
        out[f"recall_ge{T}"]=float((pr[haz]>=T).mean()) if haz.sum() else None
        out[f"fpr_ge{T}"]  =float((pr[nz]>=T).mean())  if nz.sum()  else None
        out[f"n_ge{T}"]=int(haz.sum())
    return out
def m_condition(p,y):
    yb=y.astype(int); return dict(auroc=roc_auc_score(yb,p) if len(set(yb))>1 else float("nan"), f1=f1_score(yb,(p>0.5).astype(int),zero_division=0))

def evaluate(method, uniq,g,log,lab):
    return dict(material=m_material(aggregate(method,uniq,g,log,"material"),lab["material"]),
                route   =m_route(   aggregate(method,uniq,g,log,"route"),   lab["route"]),
                threat  =m_threat(  aggregate(method,uniq,g,log,"threat"),  lab["threat"]),
                condition=m_condition(aggregate(method,uniq,g,log,"condition"),lab["condition"]))


## 1. Parameter-free aggregations on the 457-item k-fold pool (the trustworthy result)


In [ ]:
import pandas as pd
uniq,g,log,lab=group(KF,"test")
print(f"k-fold: {sum(len(v) for v in g.values())} views over {len(uniq)} items (mean {np.mean([len(v) for v in g.values()]):.1f}/item)")
print("threat label counts:", {int(t):int((lab['threat']==t).sum()) for t in sorted(set(lab['threat']))})
METHODS=["mean_logit","mean_prob","conf_weighted","best_view","max_noisyOR","median_logit","lse"]
res={m:evaluate(m,uniq,g,log,lab) for m in METHODS}
rows=[]
for m in METHODS:
    r=res[m]; rows.append(dict(method=m, matF1=r["material"]["f1"], matAcc=r["material"]["acc"],
        routeF1=r["route"]["f1"], thrQWK=r["threat"]["qwk"], thrMAE=r["threat"]["mae"],
        hazRec_ge2=r["threat"]["recall_ge2"], hazFPR_ge2=r["threat"]["fpr_ge2"], condAUROC=r["condition"]["auroc"]))
df=pd.DataFrame(rows).set_index("method").round(3);
import IPython.display as D; D.display(df)
json.dump(res, open(f"{OUT}/view_aggregation_kfold.json","w"), indent=2)
df.to_csv(f"{OUT}/view_aggregation_kfold.csv"); print("saved JSON + CSV to", OUT)


## 2. Safety view: hazard recall vs false-alarm trade-off

Your system's priority is **never miss a hazard**. `mean` aggregation can *dilute* a hazard that
only one view sees. `max` / **noisy-OR** ("if any view flags high threat, flag the item") targets
recall directly -- at a false-positive (over-routing to Special) cost. Below: bootstrap 95% CIs and
the recall/FPR trade-off, so the choice is a documented policy decision, not a guess.


In [ ]:
def boot_ci(method, metric_fn, B=2000, seed=0):
    rng=np.random.default_rng(seed); n=len(uniq)
    base={h:aggregate(method,uniq,g,log,h) for h in HEADS}
    vals=[metric_fn({h:base[h][s] for h in HEADS},{h:lab[h][s] for h in HEADS}) for s in (rng.integers(0,n,n) for _ in range(B))]
    vals=np.array([v for v in vals if v==v]); return float(np.percentile(vals,2.5)), float(np.percentile(vals,97.5))

haz2=lambda p,y: (m_threat(p["threat"],y["threat"])["recall_ge2"] or np.nan)
matf=lambda p,y: m_material(p["material"],y["material"])["f1"]
print("material F1  mean_logit  95% CI:", tuple(round(x,3) for x in boot_ci("mean_logit",matf)))
for m in ["mean_logit","median_logit","lse","max_noisyOR"]:
    ci=boot_ci(m,haz2); r=res[m]["threat"]
    print(f"hazard recall>=2  {m:12s}: {r['recall_ge2']:.3f}  CI[{ci[0]:.3f},{ci[1]:.3f}]   FPR {r['fpr_ge2']:.3f}")


In [ ]:
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig,ax=plt.subplots(1,2,figsize=(12,4.2))
    x=np.arange(len(METHODS))
    ax[0].bar(x,[res[m]["material"]["f1"] for m in METHODS],width=.38,label="material F1",color="#E65100")
    ax[0].bar(x+.4,[res[m]["route"]["f1"] for m in METHODS],width=.38,label="route F1",color="#2E7D32")
    ax[0].set_xticks(x+.2); ax[0].set_xticklabels(METHODS,rotation=40,ha="right"); ax[0].legend(); ax[0].set_title("material / route F1 by aggregation")
    for m in METHODS:
        ax[1].scatter(res[m]["threat"]["fpr_ge2"],res[m]["threat"]["recall_ge2"],s=60)
        ax[1].annotate(m,(res[m]["threat"]["fpr_ge2"],res[m]["threat"]["recall_ge2"]),fontsize=8,xytext=(4,-2),textcoords="offset points")
    ax[1].set_xlabel("hazard FPR (>=2)"); ax[1].set_ylabel("hazard recall (>=2)"); ax[1].set_title("threat: recall vs false-alarm")
    plt.tight_layout(); plt.savefig(f"{OUT}/view_aggregation_plot.png",dpi=150); print("saved plot"); plt.show()
except Exception as e: print("plot skipped:",e)


## 3. Learned aggregators (attention / DeepSets) -- fit on SC12 val, report on test

Do learned, parameter-*rich* aggregators beat the simple ones? On ~46 validation items the honest
expectation is **no** (they overfit). Fit on val, evaluate on test (no leakage), compared against
`mean_logit` on the *same* split.


In [ ]:
try:
    import torch, torch.nn as nn
    def g2(split):
        items=SC12[f"{split}_items"]; lg={h:SC12[f"{split}_log_{h}"] for h in HEADS}; ys={h:SC12[f"{split}_y_{h}"] for h in HEADS}
        uq=list(dict.fromkeys(items.tolist())); gd={it:[] for it in uq}
        for j,it in enumerate(items): gd[it].append(j)
        Vs=[]; lb={h:[] for h in HEADS}
        for it in uq:
            idx=gd[it]; v=np.concatenate([lg[h][idx].reshape(len(idx),-1) for h in HEADS],1)
            Vs.append(torch.tensor(v,dtype=torch.float32))
            for h in HEADS: lb[h].append(ys[h][idx[0]])
        return uq,Vs,{h:np.array(lb[h]) for h in HEADS}
    uv,Vv,Lv=g2("val"); ut,Vt,Lt=g2("test"); sl={"material":slice(0,10),"condition":slice(10,11),"route":slice(11,15),"threat":slice(15,18)}
    class Attn(nn.Module):
        def __init__(s,d=18,h=32): super().__init__(); s.f=nn.Sequential(nn.Linear(d,h),nn.GELU(),nn.Dropout(0.3),nn.Linear(h,1))
        def forward(s,v): w=torch.softmax(s.f(v),0); return (w*v).sum(0)
    class DeepSets(nn.Module):
        def __init__(s,d=18,h=32): super().__init__(); s.phi=nn.Sequential(nn.Linear(d,h),nn.GELU()); s.rho=nn.Sequential(nn.Linear(h,h),nn.GELU(),nn.Dropout(0.3),nn.Linear(h,d))
        def forward(s,v): return s.rho(s.phi(v).mean(0))
    def loss_i(o,lab,i):
        z={h:o[sl[h]] for h in HEADS}
        t=lab["threat"][i]; tgt=torch.tensor([1.0 if t>k else 0.0 for k in range(3)])
        return (nn.functional.cross_entropy(z["material"][None],torch.tensor([lab["material"][i]]))
               +0.5*nn.functional.binary_cross_entropy_with_logits(z["condition"],torch.tensor([float(lab["condition"][i])]))
               +nn.functional.cross_entropy(z["route"][None],torch.tensor([lab["route"][i]]))
               +nn.functional.binary_cross_entropy_with_logits(z["threat"],tgt))
    def cornp(l):
        c=np.cumprod(1/(1+np.exp(-l)),1); p=np.zeros((len(l),4)); p[:,0]=1-c[:,0]
        for k in range(1,3): p[:,k]=c[:,k-1]-c[:,k]
        p[:,-1]=c[:,-1]; return p
    def sm(z): z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
    def scored(O):
        return dict(matF1=round(f1_score(Lt["material"],sm(O[:,0:10]).argmax(1),average="macro",labels=range(10),zero_division=0),3),
                    routeF1=round(f1_score(Lt["route"],sm(O[:,11:15]).argmax(1),average="macro",labels=range(4),zero_division=0),3),
                    thrQWK=round(cohen_kappa_score(Lt["threat"],cornp(O[:,15:18]).argmax(1),weights="quadratic",labels=list(range(4))),3),
                    condAUROC=round(roc_auc_score(Lt["condition"].astype(int),1/(1+np.exp(-O[:,10]))),3))
    def fit(Model):
        torch.manual_seed(0); m=Model(); opt=torch.optim.AdamW(m.parameters(),lr=3e-3,weight_decay=1e-2)
        for ep in range(120):
            for i in np.random.default_rng(ep).permutation(len(Vv)):
                opt.zero_grad(); loss_i(m(Vv[i]),Lv,i).backward(); opt.step()
        m.eval()
        with torch.no_grad(): return scored(torch.stack([m(Vt[i]) for i in range(len(Vt))]).numpy())
    base=scored(np.stack([Vt[i].mean(0).numpy() for i in range(len(Vt))]))
    print("SC12 test (46 items) -- learned aggregators vs baseline:")
    print("  baseline mean_logit:",base)
    print("  learned  attention :",fit(Attn))
    print("  learned  deepsets  :",fit(DeepSets))
    print("  (small val set -> learned methods typically overfit and lose; that is the expected finding)")
except Exception as e: print("learned section skipped:",e)


## 4. Recommendation -- best aggregation per head


In [ ]:
prim={"material":("material","f1"),"route":("route","f1"),"threat":("threat","qwk"),"condition":("condition","auroc")}
print("Best parameter-free aggregation per head (457-item k-fold), vs current mean_logit:\n")
best={}
for head,(sec,key) in prim.items():
    scoreof={m:res[m][sec][key] for m in METHODS}
    bm=max(scoreof,key=scoreof.get); best[head]=bm
    print(f"  {head:9s}: best = {bm:12s} {scoreof[bm]:.3f}   (mean_logit {scoreof['mean_logit']:.3f}, delta {scoreof[bm]-scoreof['mean_logit']:+.3f})")
print("\nSafety (hazard recall>=2): mean_logit %.3f | median_logit %.3f (free) | lse %.3f | max/noisyOR %.3f (max recall, higher FPR)"%(
    res['mean_logit']['threat']['recall_ge2'],res['median_logit']['threat']['recall_ge2'],res['lse']['threat']['recall_ge2'],res['max_noisyOR']['threat']['recall_ge2']))
rec=dict(per_head_best=best,
         recommended=dict(material="mean_logit",route="mean_prob_or_best_view",threat_quality="median_logit",
                          threat_safety="max_noisyOR (policy: maximize hazard recall) or median_logit (free, no FPR cost)"),
         note="Learned aggregators overfit the small val set and do not beat simple pooling.")
json.dump(rec, open(f"{OUT}/view_aggregation_recommendation.json","w"), indent=2)
print("\nsaved", f"{OUT}/view_aggregation_recommendation.json")


## How to read this / what to put in the report

- **Material** is already optimal with `mean_logit` -- no change; a good "we validated the default" point.
- **Route** gains a little from `mean_prob` / `best_view` (within the wide route CI -- report as an ablation, not a headline).
- **Threat / safety** is where aggregation matters: `median_logit` gives a small **free** hazard-recall
  bump (no FPR cost), and **noisy-OR** pushes hazard recall to ~0.99 at a documented false-alarm cost --
  a legitimate *policy knob* for a "never miss a hazard" system, backed by bootstrap CIs.
- **Learned aggregators lose** on this data size -- an honest negative result that strengthens the
  "simple, parameter-free fusion is the right choice for a small dataset" argument.

All numbers come from cached logits only; nothing was retrained, so this is fully reproducible.
